# Laplace and CVI: A Pendulum Filter

This example is part of the *Universal Inference* section, which explores running inference in a broad class of models by approximating what has no closed form. Here the nonlinearities are functions in the model, deterministic nodes, and the fallback computes them as well: no approximation method is chosen per node.

The model is the noisy pendulum of Särkkä's *Bayesian Filtering and Smoothing*, a standard test of nonlinear filters. The state is the angle $\theta$ and the angular velocity $\omega$, discretised with step $\Delta t$, and a sensor measures the horizontal position of the bob, $\sin\theta$:

$$\begin{aligned}
x_t &= \begin{pmatrix} \theta_t \\ \omega_t \end{pmatrix} \sim \mathcal{N}\left(f(x_{t-1}), Q\right), \qquad f(x) = \begin{pmatrix} \theta + \omega\,\Delta t \\ \omega - g \sin(\theta)\,\Delta t \end{pmatrix},\\
y_t &\sim \mathcal{N}(\sin\theta_t, r).
\end{aligned}$$

We filter it online, one observation at a time, three ways: with `fallback = LaplaceApproximation()`, with `fallback = CVI()`, and with RxInfer's unscented transform chosen by hand for both functions, the usual way. A particle filter is the reference.

In [ ]:
using RxInfer, BayesBase, Distributions, LinearAlgebra, StableRNGs
using Plots, PrettyTables, Printf

## Data

In [ ]:
const Δt, g = 0.05, 9.81
transition(x) = [x[1] + x[2] * Δt, x[2] - g * sin(x[1]) * Δt]
sine(x) = sin(x[1])

qc, r, T = 0.5, 0.1, 100
Q = qc * [Δt^3/3 Δt^2/2; Δt^2/2 Δt]

rng = StableRNG(7)
x_true = Vector{Vector{Float64}}(undef, T)
x_true[1] = [1.5, 0.0]
for t in 2:T
    x_true[t] = rand(rng, MvNormal(transition(x_true[t - 1]), Q))
end
y = [rand(rng, Normal(sine(x), sqrt(r))) for x in x_true]
θ_true = first.(x_true)

plot(θ_true; color = :black, label = "angle θₜ", xlabel = "t", size = (800, 350))
scatter!(y; color = :gray, markersize = 2, label = "observations yₜ of sin θₜ")
hline!([-π / 2, π / 2]; color = :gray, linestyle = :dot, label = "±π/2")

The pendulum starts at 1.5 rad and the process noise pumps energy into it: it swings past $\pm\pi/2$, where $\sin\theta$ turns back. There, one observation fits two angles, $\theta$ and $\pm\pi - \theta$, and only the dynamics tell them apart.

## The filter

Each step is a small model: the previous state's belief as a prior, the transition, and the observation. Both functions enter the model as they are; without an approximation method, they go to the fallback. Its message towards a function's output is the moment-matched Gaussian of the inputs pushed through it, and its message back towards an input is the output's message pulled back through the function. The belief about $x_t$, the transition's Gaussian times the pulled-back observation, is then projected onto a Gaussian.

In [ ]:
@model function pendulum_step(y, m, V, Q, r)
    x_prev ~ MvNormalMeanCovariance(m, V)
    x ~ MvNormalMeanCovariance(transition(x_prev), Q)
    s := sine(x)
    y ~ NormalMeanVariance(s, r)
end

carry = @autoupdates begin
    m, V = mean_cov(q(x))
end

initial = @initialization begin
    q(x) = MvNormalMeanCovariance([1.5, 0.0], diagm([0.1, 0.1]))
end

function run_filter(; kwargs...)
    observations = from(y) |> map(NamedTuple{(:y,), Tuple{Float64}}, (v) -> (y = v,))
    engine = infer(;
        model = pendulum_step(Q = Q, r = r), datastream = observations,
        autoupdates = carry, initialization = initial,
        returnvars = (:x,), historyvars = (x = KeepLast(),), keephistory = T, kwargs...,
    )
    return engine
end

The filter asks for the belief about $x_t$ only. The beliefs about the previous state and the transition's output would need messages backwards through the transition. Those are log-densities: the pulled-back observation alone says nothing about the velocity, and no proper Gaussian represents it. Universal inference does not compute them yet, so this example filters and does not smooth.

In [ ]:
laplace = run_filter(fallback = LaplaceApproximation())
cvi = run_filter(fallback = CVI(), report = true)
unscented = run_filter(algorithm = @algorithm begin
    transition() -> Unscented()
    sine() -> Unscented()
end)

angles(engine) = (first.(mean.(engine.history[:x])), [sqrt(cov(q)[1, 1]) for q in engine.history[:x]])
m_laplace, s_laplace = angles(laplace)
m_cvi, s_cvi = angles(cvi)
m_unscented, s_unscented = angles(unscented)

The report of the CVI filter, over its hundred steps, shows the two functions computed by the fallback and the belief about $x_t$ projected at every step:

In [ ]:
report_cvi = cvi.model.metadata[:report]
@assert isempty(RxInfer.diagnostics(report_cvi)) # hide
report_cvi

## The particle filter

A bootstrap particle filter with 20 000 particles: each particle moves with the dynamics, is weighted by the observation, and the cloud is resampled.

In [ ]:
function particle_filter(y; n = 20_000, rng = StableRNG(1))
    particles = rand(rng, MvNormal([1.5, 0.0], diagm([0.1, 0.1])), n)
    L = cholesky(Q).L
    means, sds = zeros(length(y)), zeros(length(y))
    for t in eachindex(y)
        if t > 1
            for j in 1:n
                particles[:, j] = transition(particles[:, j]) + L * randn(rng, 2)
            end
        end
        logw = [logpdf(Normal(sin(particles[1, j]), sqrt(r)), y[t]) for j in 1:n]
        w = exp.(logw .- maximum(logw))
        w ./= sum(w)
        means[t] = sum(w .* particles[1, :])
        sds[t] = sqrt(sum(w .* (particles[1, :] .- means[t]) .^ 2))
        particles = particles[:, rand(rng, Categorical(w), n)]
    end
    return means, sds
end

m_pf, s_pf = particle_filter(y)

## Results

In [ ]:
plt = plot(m_pf; ribbon = 2s_pf, fillalpha = 0.2, color = :gray, label = "particle filter (±2 sd)", xlabel = "t", ylabel = "angle θₜ", size = (900, 500), legend = :outerbottom, legendcolumns = 3)
plot!(plt, θ_true; color = :black, linewidth = 1, label = "true θₜ")
plot!(plt, m_unscented; color = :green, label = "unscented, by hand")
plot!(plt, m_laplace; color = :blue, label = "LaplaceApproximation()")
plot!(plt, m_cvi; color = :red, linestyle = :dash, label = "CVI()")
hline!(plt, [-π / 2, π / 2]; color = :gray, linestyle = :dot, label = "±π/2")

In [ ]:
rms(a) = sqrt(sum(abs2, a) / length(a))
coverage(m, s) = sum(abs.(m - θ_true) .<= 2s) / T
rows = [
    "particle filter" rms(m_pf - θ_true) 0.0 coverage(m_pf, s_pf);
    "unscented, by hand" rms(m_unscented - θ_true) rms(m_unscented - m_pf) coverage(m_unscented, s_unscented);
    "CVI()" rms(m_cvi - θ_true) rms(m_cvi - m_pf) coverage(m_cvi, s_cvi);
    "LaplaceApproximation()" rms(m_laplace - θ_true) rms(m_laplace - m_pf) coverage(m_laplace, s_laplace)
]
@assert rms(m_cvi - m_pf) < 0.15 && rms(m_unscented - m_pf) < 0.15 # hide
@assert rms(m_laplace - m_pf) > 1.0 # hide
@assert abs(rms(m_cvi - θ_true) - rms(m_pf - θ_true)) < 0.05 # hide
pretty_table(rows; column_labels = ["", "RMSE against the truth", "RMS difference from the particle filter", "θₜ within ±2 sd"], formatters = [fmt__printf("%.3f", 2:4)], fit_table_in_display_horizontally = false)

`CVI()` follows the particle filter closely, as the unscented filter does, with no method chosen for either function. `LaplaceApproximation()` follows it while the swing goes out past $-\pi/2$, and is lost for good as it comes back:

In [ ]:
out = findfirst(<(-π / 2), m_pf)
back = findnext(>(-π / 2), m_pf, out)
lost = findfirst(t -> abs(m_laplace[t] - m_pf[t]) > 0.5, 1:T)
@assert !isnothing(lost) && abs(lost - back) <= 2 # hide
@assert all(t -> abs(m_cvi[t] - m_pf[t]) < 0.5, 1:T) # hide
@printf("the angle goes past -π/2 at t = %d and comes back at t = %d; Laplace leaves the particle filter at t = %d\n", out, back, lost)

Near $-\pi/2$, an observation of $\sin\theta$ fits the angle and its mirror image $-\pi - \theta$ about equally, and on the way back the two are close. The belief about $x_t$ can then be skewed towards the mirror image, or have a second mode there. The Laplace method takes the Gaussian at the mode it finds and its curvature, and once that mode is on the wrong side of the turn, the next step starts from it: the filter swings the wrong way and never returns. `CVI()` fits the Gaussian over the whole belief, weighing the region the dynamics predict, so a skewed belief moves its mean a little rather than flipping it.

## References

- S. Särkkä (2013). *Bayesian Filtering and Smoothing*. Cambridge University Press.
- N. J. Gordon, D. J. Salmond and A. F. M. Smith (1993). Novel approach to nonlinear/non-Gaussian Bayesian state estimation. *IEE Proceedings F*, 140(2), 107–113.